In [2]:
%pip install pandas numpy ucimlrepo openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

print("Downloading dataset from UCI repository, please wait...")

# Fetching Online Retail dataset from UCI server
online_retail = fetch_ucirepo(id=352)
df_raw = online_retail.data.original.copy()

print("Dataset loaded successfully!")
print(f"Total initial rows: {len(df_raw)}")
df_raw.head()

Dataset loaded successfully!
Total initial rows: 541909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [9]:
print("--- STEP 2: TEXT STANDARDIZATION & DATA AUDIT ---")

# Copy raw data to keep original safe
df_clean = df_raw.copy()

# Standardize text columns by stripping spaces and fixing casing
df_clean['Country'] = df_clean['Country'].str.strip().str.title()
df_clean['Description'] = df_clean['Description'].str.strip()

# Check missing values in each column
print("\nMissing values count per column:")
print(df_clean.isnull().sum())

print(f"\nTotal duplicate rows: {df_clean.duplicated().sum()}")

--- STEP 2: TEXT STANDARDIZATION & DATA AUDIT ---

Missing values count per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Total duplicate rows: 5268


In [10]:
print("--- STEP 3: HANDLING MISSING VALUES & DUPLICATES ---")

# Remove rows where CustomerID is missing (essential for customer retention tracking)
missing_cust_count = df_clean['CustomerID'].isnull().sum()
df_clean = df_clean.dropna(subset=['CustomerID'])

# Fill missing product descriptions with a default label
df_clean['Description'] = df_clean['Description'].fillna('Unknown Product')

# Remove exact duplicate transaction rows
duplicates_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

print(f"Removed rows with missing CustomerID: {missing_cust_count}")
print(f"Removed duplicate rows: {duplicates_count}")
print(f"Rows remaining: {len(df_clean)}")

--- STEP 3: HANDLING MISSING VALUES & DUPLICATES ---
Removed rows with missing CustomerID: 135080
Removed duplicate rows: 5225
Rows remaining: 401604


In [11]:
print("--- STEP 4: TYPE & RANGE VALIDATION ---")

# Fix data types for analysis
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
df_clean['CustomerID'] = df_clean['CustomerID'].astype(float).astype(int).astype(str)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Remove cancelled invoices (Invoice numbers starting with 'C')
cancellation_mask = df_clean['InvoiceNo'].str.startswith('C')
cancellation_count = cancellation_mask.sum()
df_clean = df_clean[~cancellation_mask]

# Remove rows with zero or negative Quantity and UnitPrice
invalid_range_mask = (df_clean['Quantity'] <= 0) | (df_clean['UnitPrice'] <= 0)
invalid_range_count = invalid_range_mask.sum()
df_clean = df_clean[~invalid_range_mask]

print(f"Removed cancelled orders: {cancellation_count}")
print(f"Removed invalid quantity/price records: {invalid_range_count}")
print(f"Rows remaining: {len(df_clean)}")

--- STEP 4: TYPE & RANGE VALIDATION ---
Removed cancelled orders: 8872
Removed invalid quantity/price records: 40
Rows remaining: 392692


In [12]:
print("--- STEP 5: REMOVING NON-PRODUCT STOCKCODES ---")

# List of administrative non-product codes (postage, manual fees, bank charges)
non_product_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'PADS']
non_product_mask = df_clean['StockCode'].isin(non_product_codes)
non_product_count = non_product_mask.sum()

# Filter out non-product entries
df_clean = df_clean[~non_product_mask]

print(f"Removed non-product administrative records: {non_product_count}")
print(f"Rows remaining: {len(df_clean)}")

--- STEP 5: REMOVING NON-PRODUCT STOCKCODES ---
Removed non-product administrative records: 1409
Rows remaining: 391283


In [13]:
print("--- STEP 6: DATE RANGE VALIDATION ---")

# Define official UCI dataset timeframe
start_date = '2010-12-01'
end_date = '2011-12-09'

date_range_mask = (df_clean['InvoiceDate'] >= start_date) & (df_clean['InvoiceDate'] <= end_date)
invalid_date_count = (~date_range_mask).sum()

# Keep only valid date records
df_clean = df_clean[date_range_mask]

print(f"Removed out-of-range date records: {invalid_date_count}")
print(f"Rows remaining: {len(df_clean)}")

--- STEP 6: DATE RANGE VALIDATION ---
Removed out-of-range date records: 605
Rows remaining: 390678


In [14]:
print("--- STEP 7: OUTLIER DETECTION (IQR METHOD) ---")

# Calculate Total Price for each item
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Apply Interquartile Range (IQR) on TotalPrice
Q1 = df_clean['TotalPrice'].quantile(0.25)
Q3 = df_clean['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (df_clean['TotalPrice'] < lower_bound) | (df_clean['TotalPrice'] > upper_bound)
outlier_count = outlier_mask.sum()

# Filter out extreme outliers
df_clean = df_clean[~outlier_mask]

print(f"Removed extreme price/total outliers: {outlier_count}")
print(f"Rows remaining: {len(df_clean)}")

--- STEP 7: OUTLIER DETECTION (IQR METHOD) ---
Removed extreme price/total outliers: 30531
Rows remaining: 360147


In [17]:
print("--- STEP 8: MEMORY OPTIMIZATION & EXPORT ---")

# Optimize memory usage for faster processing
df_clean['Quantity'] = df_clean['Quantity'].astype('int32')
df_clean['UnitPrice'] = df_clean['UnitPrice'].astype('float32')

# Save final highly-refined dataset to CSV
df_clean.to_csv("Cleaned_Online_Retail.csv", index=False)

# Create comprehensive summary table for project report / presentation
summary_report = pd.DataFrame({
    "Preprocessing & Validation Step": [
        "1. Raw Dataset Loaded",
        "2. Text Standardization (Country/Desc)",
        "3. Missing CustomerID Removed",
        "4. Duplicate Records Removed",
        "5. Cancelled Orders ('C') Removed",
        "6. Range Validation (Quantity/Price > 0)",
        "7. Non-Product Codes Removed (POST/DOT)",
        "8. Date Range Validation Checked",
        "9. Advanced Outlier Removal (IQR)",
        "10. Final Analysis-Ready Dataset"
    ],
    "Record Count": [
        len(df_raw),
        len(df_raw),
        len(df_raw) - missing_cust_count,
        len(df_raw) - missing_cust_count - duplicates_count,
        len(df_clean) + outlier_count + invalid_date_count + non_product_count + invalid_range_count,
        len(df_clean) + outlier_count + invalid_date_count + non_product_count,
        len(df_clean) + outlier_count + invalid_date_count,
        len(df_clean) + outlier_count,
        len(df_clean),
        len(df_clean)
    ]
})

print("\n🎉 Module 2 completed successfully! Cleaned_Online_Retail.csv saved.\n")
display(summary_report)

--- STEP 8: MEMORY OPTIMIZATION & EXPORT ---

🎉 Module 2 completed successfully! Cleaned_Online_Retail.csv saved.



,Preprocessing & Validation Step,Record Count
0,1. Raw Dataset Loaded,541909
1,2. Text Standardization (Country/Desc),541909
2,3. Missing CustomerID Removed,406829
3,4. Duplicate Records Removed,401604
4,5. Cancelled Orders ('C') Removed,392732
5,6. Range Validation (Quantity/Price > 0),392692
6,7. Non-Product Codes Removed (POST/DOT),391283
7,8. Date Range Validation Checked,390678
8,9. Advanced Outlier Removal (IQR),360147
9,10. Final Analysis-Ready Dataset,360147
